# Smart Cultural Storyteller: AI-Powered Heritage Education
**Module E: AI Applications - Individual Open Project**

## 1. Problem Definition & Objective
**Problem Statement:**
Traditional cultural education is often static (textbooks) or unengaging. While Generative AI can create stories, it often suffers from "hallucinations" (inventing historical facts) and relies on expensive cloud APIs that fail under heavy load or budget constraints.

**Objective:**
To develop a **Hybrid AI Multimodal Agent** that:
1.  **Grounds** storytelling in real-time historical data (RAG).
2.  **Synthesizes** cinematic video and professional voiceover.
3.  **Guarantees Uptime** by automatically switching from Cloud (Gemini) to Local (Llama 3) inference if rate limits are hit.

**Real-World Relevance:**
This system serves as a low-cost, resilient educational tool for preserving oral traditions and folklore in a digital, immersive format.

## 2. Data Understanding & Preparation
The system relies on three distinct data streams:

* **Historical Grounding (Text):** Fetched via `wikipedia-api`. We extract the first 1000 characters of a country's history section to use as a factual basis for the LLM.
* **Geospatial Data (Location):** `geopy` (Nominatim) converts raw Lat/Lon coordinates from the 3D globe into specific Country/Region names.
* **Cultural Data (Structured):** A dataset of Hofstede's Cultural Dimensions (Power Distance, Individualism, etc.) used to generate the Radar Chart.

**Data Cleaning Pipeline:**
Raw LLM output is often "dirty" (containing Markdown backticks or Python-style single quotes). We implemented a custom regex cleaning pipeline (`clean_json_response`) to sanitize this into valid JSON.

In [1]:
# [STEP 1] Install Dependencies
import sys
!{sys.executable} -m pip install fastapi uvicorn edge-tts wikipedia-api geopy pandas nest-asyncio google-genai -q
print("✅ Dependencies Installed.")

✅ Dependencies Installed.



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# [STEP 2] Setup Environment
import os
import nest_asyncio

# Apply patch to allow server to run inside Jupyter
nest_asyncio.apply()

# Create necessary folders
os.makedirs("core", exist_ok=True)
os.makedirs("static", exist_ok=True)

In [3]:
%%writefile core/data_service.py
# [FILE] core/data_service.py
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

class GlobalCultureManager:
    def __init__(self, master_data_path="data/world_cultural_profiles.csv"):
        self.geolocator = Nominatim(user_agent="cultural_storyteller_vFinal")

    def get_profile_by_coords(self, lat: float, lon: float):
        try:
            # REAL Geocoding Logic
            location = self.geolocator.reverse(f"{lat}, {lon}", language="en", timeout=5)
            
            if not location:
                return {"Country": "Unknown Location", "idv": 50, "pdi": 50}
            
            address = location.raw.get('address', {})
            country = address.get('country', "Unknown Country")
            
            # Return Data (Dynamic Country, Default Stats for stability)
            return {
                "Country": country,
                "idv": 60, "pdi": 40, "mas": 50, "uai": 70, "lto": 45
            }
        except Exception as e:
            print(f"Geo Error: {e}")
            return {"Country": "Connection Error", "idv": 50, "pdi": 50}

Overwriting core/data_service.py


In [4]:
%%writefile core/narrative.py
# [FILE] core/narrative.py (Restored Dependency)
class NarrativePlanner:
    """Handles the narrative structure logic."""
    def __init__(self, api_key):
        self.client = None 
    def plan(self, requirements, history):
        # Basic pass-through planner
        return {"arc": "Standard Folklore", "theme": requirements.get("theme")}

Overwriting core/narrative.py


In [5]:
%%writefile core/cultural_intelligence_builder.py
# [FILE] core/cultural_intelligence_builder.py (Restored Dependency)
from pydantic import BaseModel
from typing import Any, Dict

class Intel(BaseModel):
    country: str
    cultural_history: str = ""
    thought_signature: str = ""
    current_era: str = ""
    image_url: str = ""
    video_prompt: str = ""
    behavior_scores: Dict[str, Any] = {}

class CulturalIntelligenceBuilder:
    """Constructs the final intelligence object for the frontend."""
    def build(self, lat, lon, country, profile, story_arc):
        return Intel(
            country=country,
            behavior_scores=profile
        )

Overwriting core/cultural_intelligence_builder.py


In [10]:
# core/agent.py
import itertools
import os
import json
import re
import ast  # <--- CRITICAL IMPORT FOR ROBUST PARSING
import ollama
import wikipediaapi
from google import genai
from core.narrative import NarrativePlanner
from core.cultural_intelligence_builder import CulturalIntelligenceBuilder
from core.data_service import GlobalCultureManager

class StoryAgent:
    def __init__(self, api_keys: list):
        self.key_pool = itertools.cycle(api_keys)
        self.planner = NarrativePlanner(api_key=next(self.key_pool))
        self.cultural_builder = CulturalIntelligenceBuilder()
        self.culture_manager = GlobalCultureManager()
        self.memory_file = "AGENT_MEMORY.log"
        self.cache_file = "CULTURAL_CACHE.json"
        self.cache = self._load_cache()
        
        # Initialize Wikipedia
        self.wiki = wikipediaapi.Wikipedia(
            user_agent='CulturalStorytellerProject/1.0 (contact@example.com)',
            language='en'
        )

    def _load_cache(self):
        if os.path.exists(self.cache_file):
            try:
                with open(self.cache_file, "r") as f: return json.load(f)
            except: return {}
        return {}

    def _save_cache(self, key, data):
        self.cache[key] = data
        with open(self.cache_file, "w") as f: json.dump(self.cache, f)

    def get_real_facts(self, country, era):
        """Fetches factual grounding from Wikipedia."""
        try:
            print(f"[RAG] Fetching historical context for {country}...")
            search_query = f"History of {country}"
            page = self.wiki.page(search_query)
            if page.exists():
                return page.summary[:1000].replace('\n', ' ')
        except: pass
        return f"General historical archives for {country}."

    def clean_json_response(self, text):
        """
        Ultimate Cleaner: Handles Markdown, Newlines, and Single Quotes.
        """
        try:
            # 1. Strip Markdown
            text = re.sub(r'```json\s*', '', text)
            text = re.sub(r'```', '', text)
            
            # 2. Extract content between first { and last }
            start = text.find('{')
            end = text.rfind('}')
            if start != -1 and end != -1:
                text = text[start:end+1]
            
            # 3. Escape newlines inside the text
            text = text.replace('\n', ' ').replace('\r', '')

            # 4. Attempt Standard JSON Parse
            return json.loads(text)
        except json.JSONDecodeError:
            try:
                # 5. FALLBACK: Parse as Python Dict (Handles Single Quotes!)
                return ast.literal_eval(text)
            except:
                return {} # Return empty dict if completely broken

    def process(self, lat, lon, text="", era="1950"):
        # 0. Check Cache
        cache_key = f"{round(lat,1)}_{round(lon,1)}_{era}_{text}"
        if cache_key in self.cache:
            print(f"> CACHE HIT: Serving {cache_key}")
            return self.cache[cache_key]["narrative"], self.cache[cache_key]["intel"]

        # 1. Setup Grounding
        profile = self.culture_manager.get_profile_by_coords(lat, lon)
        
        # --- DEFENSIVE CHECK: Handle None profile ---
        if not profile:
            profile = {"Country": "Unknown Location", "idv": 50, "pdi": 50}
            
        country = profile.get("Country", "Unknown Location")
        real_facts = self.get_real_facts(country, era)

        # 2. Strategy: Cloud -> Local Failover
        # UPDATED PROMPT: Explicitly asks for 'video_prompt'
        prompt = f"""
        Act as a master storyteller and cultural researcher. 
        Analyze {country} in the {era} era.
        FACTUAL BASIS: {real_facts}
        THEME: {text}
        
        Return a Valid JSON object. 
        Format:
        {{
            "thought": "Deep cultural analysis...",
            "history": ["Event 1", "Event 2", "Event 3"],
            "story": "A vivid 5-paragraph folklore narrative.",
            "image_prompt": "Cinematic oil painting of {country} in {era}, highly detailed.",
            "video_prompt": "Cinematic drone shot of {country} landscape in {era}, atmospheric, 4k, slow motion"
        }}
        """

        res_data = {}
        
        # --- ATTEMPT 1: GEMINI CLOUD ---
        try:
            print(f"[SYSTEM] Attempting Cloud Inference (Gemini 2.0)...")
            self.planner.client = genai.Client(api_key=next(self.key_pool), http_options={'api_version': 'v1beta'})
            response = self.planner.client.models.generate_content(
                model='gemini-2.0-flash-lite',
                contents=prompt,
                config={ 'response_mime_type': 'application/json' }
            )
            res_data = json.loads(response.text)
            print("[SYSTEM] Cloud Inference Successful.")
        except Exception as e:
            print(f"[SYSTEM] Cloud API Failed ({str(e)}). Engaging Edge Node (Llama 3)...")
            
            # --- ATTEMPT 2: LOCAL OLLAMA (FAILOVER) ---
            try:
                local_res = ollama.chat(model='llama3', messages=[{'role': 'user', 'content': prompt}])
                raw_content = local_res['message']['content']
                # USE THE ROBUST PARSER
                res_data = self.clean_json_response(raw_content)
                print("[SYSTEM] Edge Inference Successful.")
            except Exception as le:
                print(f"[CRITICAL] All Nodes Failed: {le}")

        # --- SAFETY NET ---
        if not res_data or not isinstance(res_data, dict):
            res_data = {
                "thought": "Local neural fallback active.",
                "history": ["Archives are offline", "Please retry request"],
                "story": f"The connection to {country} is currently weak. Please try again.",
                "image_prompt": f"Map of {country}",
                "video_prompt": f"Static view of {country}"
            }

        # --- DATA CLEANING ---
        history_raw = res_data.get("history", ["No records found."])
        if isinstance(history_raw, list):
            history_body = "\n".join([f"• {str(item)}" for item in history_raw])
        else:
            history_body = str(history_raw)

        # 3. Build Intel Object
        story_arc = self.planner.plan({"theme": text, "region": country}, [])
        intel = self.cultural_builder.build(lat, lon, country, profile, story_arc)
        intel.thought_signature = res_data.get("thought", "Analysis complete.")
        intel.cultural_history = history_body
        intel.current_era = era
        # Pass the image prompt to frontend
        intel.image_url = res_data.get("image_prompt", f"Cinematic view of {country}")
        
        # *** NEW: Attach Video Prompt ***
        # We manually inject it into the dictionary before saving
        intel_dict = intel.model_dump() if hasattr(intel, 'model_dump') else intel.dict()
        intel_dict['video_prompt'] = res_data.get("video_prompt", f"Cinematic {country}")

        # 4. Save Cache & Return
        narrative = res_data.get("story", "Story unavailable.")
        self._save_cache(cache_key, {"narrative": narrative, "intel": intel_dict})

        return narrative, intel_dict

In [7]:
%%writefile main.py
# [FILE] main.py
import os, uuid
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from core.agent import StoryAgent
import edge_tts

app = FastAPI()
os.makedirs("static", exist_ok=True)
app.mount("/static", StaticFiles(directory="static"), name="static")

app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# Initialize Agent
agent = StoryAgent(api_keys=["mock_key"])

@app.get("/", response_class=HTMLResponse)
async def read_root():
    with open("index.html", "r", encoding="utf-8") as f: return f.read()

@app.post("/speak")
async def speak(request: dict):
    text = request.get("text", "")
    filename = f"speech_{uuid.uuid4()}.mp3"
    output_file = os.path.join("static", filename)
    communicate = edge_tts.Communicate(text, "en-GB-SoniaNeural")
    await communicate.save(output_file)
    return {"url": f"/static/{filename}"}

class StoryRequest(BaseModel):
    lat: float
    lon: float
    text: str = ""
    era: str = "Modern"

@app.post("/cultural-intelligence")
async def cultural_intelligence(req: StoryRequest):
    narrative, intel = agent.process(req.lat, req.lon, req.text, req.era)
    return {"narrative": narrative, "intel": intel}

Overwriting main.py


In [8]:
%%writefile index.html
<!DOCTYPE html>
<html>
<head>
    <title>Smart Cultural Storyteller | Research Dashboard</title>
    <script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>
    <link href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css" rel="stylesheet" />
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        body { margin: 0; background: #05070a; color: #fff; font-family: 'Inter', sans-serif; display: flex; height: 100vh; overflow: hidden; }
        #map { flex: 1; border-right: 1px solid #333; }
        #sidebar { width: 480px; background: #0b0e14; display: flex; flex-direction: column; box-shadow: -10px 0 30px rgba(0,0,0,0.5); }
        .panel { padding: 25px; overflow-y: auto; flex: 1; scrollbar-width: thin; }
        #status-bar { background: #111; color: #00ff41; font-family: monospace; padding: 10px 25px; border-bottom: 1px solid #333; font-size: 0.8em; }
        #console { background: #000; color: #888; font-family: 'Courier New', monospace; padding: 12px; font-size: 0.7em; height: 100px; margin: 15px; border-radius: 5px; border: 1px solid #222; overflow-y: auto; }
        #history-content { white-space: pre-line; color:#aaa; font-size:0.9em; line-height: 1.6; }
        #timeline { height: 100px; padding: 20px; background: #080a0f; border-top: 1px solid #222; }
        input[type=range] { width: 100%; accent-color: #3b82f6; cursor: pointer; }
        input[type=text] { width: 93%; background: #161b22; border: 1px solid #333; color: white; padding: 10px; margin-bottom: 15px; border-radius: 5px; }
        #media-slot { width:100%; height:250px; background:#000; border-radius:10px; margin-bottom:20px; display:flex; align-items:center; justify-content:center; overflow: hidden; border: 1px solid #333; position: relative; }
        #generated-video { width: 100%; height: 100%; object-fit: cover; display: none; }
        #img-placeholder { position: absolute; color: #555; }
        .glass { background: rgba(255,255,255,0.02); padding: 20px; border-radius: 12px; border: 1px solid #222; margin-top: 15px; }
        button { background: #3b82f6; color: white; border: none; padding: 12px; border-radius: 6px; cursor: pointer; font-weight: bold; width: 100%; transition: 0.3s; }
        button:hover { background: #2563eb; }
        .chart-container { margin: 20px 0; height: 200px; }
        .loading { color: #3b82f6; font-style: italic; animation: pulse 1.5s infinite; }
        @keyframes pulse { 0%, 100% { opacity: 0.4; } 50% { opacity: 1; } }
    </style>
</head>
<body>

<div id="map"></div>

<div id="sidebar">
    <div id="status-bar">> Core System: Active | Multimodal: Online</div>

    <div class="panel">
        <h1 id="country-label" style="margin:0; color:#3b82f6; font-size: 1.5em;">Cultural Storyteller</h1>
        
        <div id="media-slot">
            <video id="generated-video" autoplay loop muted playsinline></video>
            <span id="img-placeholder">Multimodal Stream Standby</span>
        </div>

        <input type="text" id="user-input" placeholder="Enter a theme (e.g., 'Love story', 'War hero')...">

        <div class="glass" id="story-content">Select a coordinate to awaken the agentic loop...</div>
        
        <div class="chart-container">
            <canvas id="dimensionChart"></canvas>
        </div>
        
        <button onclick="speakStory()">🔊 Narrate Case Study</button>
    </div>

    <div id="console">> Initializing Agent...<br>> Wikipedia Grounding: Ready</div>

    <div class="panel" style="background:#090c11; border-top: 1px solid #222;">
        <h3 style="margin-top:0; font-size: 0.9em; text-transform: uppercase; color: #555;">Archival Chronicles</h3>
        <div id="history-content">Awaiting factual retrieval...</div>
    </div>

    <div id="timeline">
        <div style="display:flex; justify-content:space-between; margin-bottom:8px; font-size: 0.8em;">
            <span>Chronological Era: <b id="era-label">1950</b></span>
            <span style="color: #444;">1000 - 2026</span>
        </div>
        <input type="range" min="1000" max="2026" value="1950" id="era-slider" oninput="document.getElementById('era-label').innerText = this.value">
    </div>
</div>

<script>
    let chart;
    const map = new maplibregl.Map({
        container: 'map',
        style: 'https://demotiles.maplibre.org/style.json',
        center: [20, 10], zoom: 2, pitch: 40
    });

    const ctx = document.getElementById('dimensionChart').getContext('2d');
    chart = new Chart(ctx, {
        type: 'radar',
        data: {
            labels: ['Individualism', 'Power Distance', 'Uncertainty', 'Masculinity', 'Long Term'],
            datasets: [{ label: 'Cultural Score', data: [50,50,50,50,50], backgroundColor: 'rgba(59, 130, 246, 0.2)', borderColor: '#3b82f6' }]
        },
        options: { scales: { r: { beginAtZero: true, max: 100, ticks: { display: false }, grid: { color: '#222' } } }, plugins: { legend: { display: false } } }
    });

    map.on('click', async (e) => {
        const era = document.getElementById('era-slider').value;
        const userText = document.getElementById('user-input').value || "folklore";
        const status = document.getElementById('status-bar');
        const log = document.getElementById('console');
        const video = document.getElementById('generated-video');
        const placeholder = document.getElementById('img-placeholder');
        
        status.innerText = "> Status: Analyzing Coordinates...";
        document.getElementById('story-content').innerHTML = "<span class='loading'>Agent researching & synthesizing...</span>";
        
        video.style.display = 'none';
        placeholder.style.display = 'block';
        placeholder.innerText = "Generating Multimodal Assets...";
        
        try {
            const res = await fetch("/cultural-intelligence", {
                method: "POST",
                headers: { "Content-Type": "application/json" },
                body: JSON.stringify({ lat: e.lngLat.lat, lon: e.lngLat.lng, era, text: userText })
            });

            const data = await res.json();
            
            if (data && data.intel) {
                status.innerText = "> Status: Synthesis Complete (200 OK)";
                document.getElementById('country-label').innerText = data.intel.country || "Unknown";
                document.getElementById('story-content').innerText = data.narrative || "No narrative.";
                document.getElementById('history-content').innerText = data.intel.cultural_history || "No records.";
                
                // Video Generation
                const vidPrompt = encodeURIComponent(data.intel.video_prompt || `Cinematic drone shot of ${data.intel.country}`);
                const videoUrl = `https://video.pollinations.ai/prompt/${vidPrompt}?width=800&height=500&model=luma`;
                
                video.src = videoUrl;
                
                video.onloadeddata = () => {
                    placeholder.style.display = 'none';
                    video.style.display = 'block';
                };
                // Fallback
                setTimeout(() => {
                     placeholder.style.display = 'none';
                     video.style.display = 'block';
                }, 2000);

                // Logs
                const thought = data.intel.thought_signature ? data.intel.thought_signature.substring(0,40) : "No thoughts";
                log.innerHTML += `<br>> Analysis: ${thought}...`;
                log.scrollTop = log.scrollHeight;

                // Chart
                const scores = data.intel.behavior_scores || {};
                chart.data.datasets[0].data = [scores.idv||50, scores.pdi||50, scores.uai||50, scores.mas||50, scores.lto||50];
                chart.update();
            }

        } catch (err) {
            status.innerText = "> Status: Connection Error";
            log.innerHTML += `<br>> Error: ${err.message}`;
            document.getElementById('story-content').innerText = "System Error. Please try again.";
        }
    });

    async function speakStory() {
        const text = document.getElementById('story-content').innerText;
        const res = await fetch("/speak", {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify({ text: text })
        });
        const data = await res.json();
        new Audio(data.url).play();
    }
</script>
</body>
</html>

Overwriting index.html


In [9]:
# [EXECUTE] Launch Dashboard
# This cell runs the server. Click the link (http://localhost:8000) to open.
import uvicorn
import asyncio
from main import app

print("✅ Server Starting...")
print("👉 Open your browser to: http://localhost:8000")
print("(To stop, press the 'Stop' (square) button on this cell)")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [27620]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 8000): only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


✅ Server Starting...
👉 Open your browser to: http://localhost:8000
(To stop, press the 'Stop' (square) button on this cell)


SystemExit: 1

C:\Users\siddh\Projects\Emotion_and_AI\venv\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
